# COG data

The data was previously downloaded via Entrez from NCBI together with TIGR and GO.

In [7]:
import pandas as pd
import requests

In [2]:
found_results_cog = pd.read_csv("../data/example_results_cog.tsv", sep="\t")

Add COG information

In [8]:
url = "https://ftp.ncbi.nlm.nih.gov/pub/COG/COG2024/data/"

with requests.get(f"{url}/cog-24.def.tab") as r:
    with open("../data/cog-24.def.tab", "wb") as f:
        f.write(r.content)

with requests.get(f"{url}/cog-24.fun.tab") as r:
    with open("../data/cog-24.fun.tab", "wb") as f:
        f.write(r.content)

In [33]:
cog_functions = pd.read_csv("../data/cog-24.fun.tab", sep="\t", header=None,
    names=["functional_category_id", "functional_group", "rgb_color", "functional_category_description"])

In [34]:
# group header rows ("1  INFORMATION STORAGE AND PROCESSING") have a number as id
is_group = cog_functions["functional_category_id"].str.isdigit()
group_names = cog_functions.loc[is_group].set_index("functional_category_id")["functional_group"]

cog_functions = cog_functions.loc[~is_group].copy()

# X has no colour in the file, so its description ended up in rgb_color
no_color = cog_functions["functional_category_description"].isna()
cog_functions.loc[no_color, "functional_category_description"] = cog_functions.loc[no_color, "rgb_color"]
cog_functions.loc[no_color, "rgb_color"] = pd.NA

cog_functions = (
    cog_functions
    .rename(columns={"functional_group": "functional_group_id", "functional_category_id": "cog_functional_category_id"})
    .assign(functional_group=lambda d: d["functional_group_id"].map(group_names))
    .astype({
        "cog_functional_category_id": "string",
        "functional_group_id": "int64",
        "functional_group": "category",
        "rgb_color": "string",
        "functional_category_description": "string",
    })
    [["cog_functional_category_id", "functional_category_description",
      "functional_group_id", "functional_group"]]
    .reset_index(drop=True)
)

In [35]:
cog_functions

,cog_functional_category_id,functional_category_description,functional_group_id,functional_group
0,J,"Translation, ribosomal structure and biogenesis",1,INFORMATION STORAGE AND PROCESSING
1,A,RNA processing and modification,1,INFORMATION STORAGE AND PROCESSING
2,K,Transcription,1,INFORMATION STORAGE AND PROCESSING
3,L,"Replication, recombination and repair",1,INFORMATION STORAGE AND PROCESSING
4,B,Chromatin structure and dynamics,1,INFORMATION STORAGE AND PROCESSING
5,D,"Cell cycle control, cell division, chromosome ...",2,CELLULAR PROCESSES AND SIGNALING
6,Y,Nuclear structure,2,CELLULAR PROCESSES AND SIGNALING
7,V,Defense mechanisms,2,CELLULAR PROCESSES AND SIGNALING
8,T,Signal transduction mechanisms,2,CELLULAR PROCESSES AND SIGNALING
9,M,Cell wall/membrane/envelope biogenesis,2,CELLULAR PROCESSES AND SIGNALING


In [36]:
cog_information = pd.read_csv("../data/cog-24.def.tab", sep="\t", header=None,
    names=["cog_id", "cog_functional_category_id", "cog_name", "gene_name", "functional_pathway", "pubmed_id", "pdb_id"])

In [37]:
cog_information["cog_functional_category_id"] = cog_information["cog_functional_category_id"].apply(lambda x: list(x.strip()))
cog_information = cog_information.explode("cog_functional_category_id")
cog_information = cog_information.merge(cog_functions, how="left", on="cog_functional_category_id")

In [38]:
cog_information

,cog_id,cog_functional_category_id,cog_name,gene_name,functional_pathway,pubmed_id,pdb_id,functional_category_description,functional_group_id,functional_group
0,COG0001,H,Glutamate-1-semialdehyde aminotransferase,HemL,Heme biosynthesis,2045363,2CFB,Coenzyme transport and metabolism,3,METABOLISM
1,COG0002,E,N-acetyl-gamma-glutamylphosphate reductase,ArgC,Arginine biosynthesis,NaN,3DR3,Amino acid transport and metabolism,3,METABOLISM
2,COG0003,P,"Anion-transporting ATPase, ArsA/GET3 family",ArsA,NaN,NaN,1F48,Inorganic ion transport and metabolism,3,METABOLISM
3,COG0004,P,Ammonia channel protein AmtB,AmtB,NaN,12023896;15876187;18362341,1U77,Inorganic ion transport and metabolism,3,METABOLISM
4,COG0005,F,Purine nucleoside phosphorylase,XapA,Purine salvage,NaN,1YQQ,Nucleotide transport and metabolism,3,METABOLISM
...,...,...,...,...,...,...,...,...,...,...
5472,COG6138,C,"Cytochrome c552 family protein, a component of...",AvECN,NaN,37290436,8E5G,Energy production and conversion,3,METABOLISM
5473,COG6139,C,"Multiheme cytochrome C, may be a component of ...",PcECN,NaN,37290436,8E5F,Energy production and conversion,3,METABOLISM
5474,COG6141,N,"Uncharacterized protein, DUF5305 family",NaN,NaN,NaN,NaN,Cell motility,2,CELLULAR PROCESSES AND SIGNALING
5475,COG6142,M,Membrane protein insertase subunit EMC6,EMC6,NaN,32439656;25869254;32459176,NaN,Cell wall/membrane/envelope biogenesis,2,CELLULAR PROCESSES AND SIGNALING


In [44]:
found_results_cog = found_results_cog.rename(columns={"cog_acc": "cog_id"})
found_results_cog = found_results_cog.merge(cog_information, how="left", on="cog_id")

In [45]:
found_results_cog

,wp_id,cog_id,cog_functional_category_id,cog_name,gene_name,functional_pathway,pubmed_id,pdb_id,functional_category_description,functional_group_id,functional_group
0,WP_000942300.1,COG0488,J,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Translation, ribosomal structure and biogenesis",1,INFORMATION STORAGE AND PROCESSING
1,WP_000942300.1,COG0488,L,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Replication, recombination and repair",1,INFORMATION STORAGE AND PROCESSING
2,WP_000602071.1,COG0488,J,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Translation, ribosomal structure and biogenesis",1,INFORMATION STORAGE AND PROCESSING
3,WP_000602071.1,COG0488,L,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Replication, recombination and repair",1,INFORMATION STORAGE AND PROCESSING
4,WP_000524836.1,COG4152,R,"ABC-type uncharacterized transport system, ATP...",YhaQ,NaN,NaN,NaN,General function prediction only,4,POORLY CHARACTERIZED
...,...,...,...,...,...,...,...,...,...,...,...
83,WP_000163735.1,COG0414,H,Panthothenate synthetase,PanC,Pantothenate/CoA biosynthesis,17968677,1DIN,Coenzyme transport and metabolism,3,METABOLISM
84,WP_000621532.1,COG0853,H,Aspartate 1-decarboxylase,PanD,Pantothenate/CoA biosynthesis,NaN,1AW8,Coenzyme transport and metabolism,3,METABOLISM
85,WP_000862727.1,COG5146,H,Pantothenate kinase,PanK,Pantothenate/CoA biosynthesis,NaN,6UJ5,Coenzyme transport and metabolism,3,METABOLISM
86,WP_000872653.1,COG0301,H,Adenylyl- and sulfurtransferase ThiI (thiamine...,ThiI,Thiamine biosynthesis,22773787,1VBK,Coenzyme transport and metabolism,3,METABOLISM


## Add this to the example data

In [46]:
test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")

/var/folders/z3/xb62blvs5xg7hy35ktdz1mdw0000gn/T/ipykernel_28135/911866719.py:1: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")


In [47]:
test_data = test_data.assign(wp_id=test_data["Majority protein IDs"].str.extract(r"(WP_[\d.]+)", expand=False))

In [48]:
test_data = test_data.merge(found_results_cog, how="left", on="wp_id")

In [49]:
test_data

,Majority protein IDs,NWMN neu,NWMN alt,gen symbol,Annotation,TIGRFAM Sub Role,The SEED,wp_id,cog_id,cog_functional_category_id,cog_name,gene_name,functional_pathway,pubmed_id,pdb_id,functional_category_description,functional_group_id,functional_group
0,WP_000942300.1|NWMN_RS07340|NWMN_1303|,NWMN_RS07340,NWMN_1303,NWMN_1303,ATP-binding cassette domain-containing protei...,0,NaN,WP_000942300.1,COG0488,J,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Translation, ribosomal structure and biogenesis",1.0,INFORMATION STORAGE AND PROCESSING
1,WP_000942300.1|NWMN_RS07340|NWMN_1303|,NWMN_RS07340,NWMN_1303,NWMN_1303,ATP-binding cassette domain-containing protei...,0,NaN,WP_000942300.1,COG0488,L,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Replication, recombination and repair",1.0,INFORMATION STORAGE AND PROCESSING
2,WP_000602071.1|NWMN_RS11270|NWMN_1954|,NWMN_RS11270,NWMN_1954,NWMN_1954,ABC-F type ribosomal protection protein[Staph...,0,NaN,WP_000602071.1,COG0488,J,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Translation, ribosomal structure and biogenesis",1.0,INFORMATION STORAGE AND PROCESSING
3,WP_000602071.1|NWMN_RS11270|NWMN_1954|,NWMN_RS11270,NWMN_1954,NWMN_1954,ABC-F type ribosomal protection protein[Staph...,0,NaN,WP_000602071.1,COG0488,L,ABC cassette proteins with duplicated ATPase d...,Uup,NaN,30597160;38349882;38943426,5ZXD,"Replication, recombination and repair",1.0,INFORMATION STORAGE AND PROCESSING
4,WP_000524836.1|NWMN_RS12920|NWMN_2242|,NWMN_RS12920,NWMN_2242,NWMN_2242,ABC transporter ATP-binding protein[Staphyloc...,Other,NaN,WP_000524836.1,COG4152,R,"ABC-type uncharacterized transport system, ATP...",YhaQ,NaN,NaN,NaN,General function prediction only,4.0,POORLY CHARACTERIZED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,WP_000396077.1|NWMN_RS11560|NWMN_1998|,NWMN_RS11560,NWMN_1998,NWMN_1998,thiaminase II[Staphylococcus aureus subsp. au...,0,"""Thiamin biosynthesis""",WP_000396077.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
199,WP_000872653.1|NWMN_RS09020|NWMN_1609|,NWMN_RS09020,NWMN_1609,thiI,tRNA uracil 4-sulfurtransferase ThiI[Staphylo...,Thiamine; tRNA and rRNA base modification,"""Thiamin biosynthesis""",WP_000872653.1,COG0301,H,Adenylyl- and sulfurtransferase ThiI (thiamine...,ThiI,Thiamine biosynthesis,22773787,1VBK,Coenzyme transport and metabolism,3.0,METABOLISM
200,WP_000872653.1|NWMN_RS09020|NWMN_1609|,NWMN_RS09020,NWMN_1609,thiI,tRNA uracil 4-sulfurtransferase ThiI[Staphylo...,Thiamine; tRNA and rRNA base modification,"""Thiamin biosynthesis""",WP_000872653.1,COG0301,J,Adenylyl- and sulfurtransferase ThiI (thiamine...,ThiI,Thiamine biosynthesis,22773787,1VBK,"Translation, ribosomal structure and biogenesis",1.0,INFORMATION STORAGE AND PROCESSING
201,WP_000594959.1|NWMN_RS11555|NWMN_1997|,NWMN_RS11555,NWMN_1997,thiD,bifunctional hydroxymethylpyrimidine kinase/p...,Thiamine,"""Thiamin biosynthesis""",WP_000594959.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN


In [50]:
test_data.to_excel("../data/example_s_aureus_cog.xlsx", index=False)